May 2025 
BC205: Algorithms for Bioinformatics.
Exercise II. Discovering and Evaluating hidden motifs in Sequences
Chairetaki Antonia

We start by reading the nucleotide sequences and for each motif length k (3-7) we iteratively refine a set of motifs. In each iteration, we remove a motif and build a probabilistic profile (PWM) matrice from the remaining motifs, and then we sample a new motif for the removed sequence based on the PWM we build. We repeat this process until we end up to a conserved pattern. Finally, we calculate the Information Content (IC) for the discovered PWMs, reporting the k and the motif with the highest IC, which is correspond to the most significant pattern found in the sequences. In our case the main motif found was GAT.

In [11]:
import random
import math
from collections import Counter, defaultdict

random.seed(42) # For reproducibility

def read_fasta(file_path):
    #Reads sequences from a FASTA-like file.
    sequences = []
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line.startswith('>') and line != "":
                sequences.append(line.upper())
    return sequences

def create_profile(motifs):
    #Builds a profile matrix from motifs using pseudocounts.
    if not motifs:
        return {'A': [], 'C': [], 'G': [], 'T': []}

    k = len(motifs[0])
    profile = defaultdict(lambda: [1.0] * k)

    for motif in motifs:
        for i, base in enumerate(motif):
            profile[base][i] += 1

    total_per_column = [len(motifs) + 4] * k

    for base in profile:
        profile[base] = [profile[base][i] / total_per_column[i] for i in range(k)]
    return profile

def kmer_prob(kmer, profile):
    #Calculates a k-mer's probability under a profile.
    prob = 1.0
    for i, base in enumerate(kmer):
        if base not in profile or i >= len(profile[base]):
            return 0.0
        prob *= profile[base][i]
    return prob

def sample_kmer(seq, k, profile):
    #Samples a k-mer from a sequence based on profile probabilities
    if len(seq) < k:
        return None

    kmer_candidates = [seq[i:i+k] for i in range(len(seq)-k+1)]
    if not kmer_candidates:
        return None

    probs = [kmer_prob(kmer, profile) for kmer in kmer_candidates]
    total_prob_sum = sum(probs)

    if total_prob_sum == 0:
        weights = [1.0 / len(kmer_candidates)] * len(kmer_candidates)
    else:
        weights = [p / total_prob_sum for p in probs]

    return random.choices(kmer_candidates, weights=weights, k=1)[0]

def score_motifs(motifs):
    #Scores motif set conservation (sum of mismatches from consensus)
    if not motifs:
        return float('inf')

    k = len(motifs[0])
    if k == 0:
        return float('inf')

    mismatch_score = 0
    
    for i in range(k):
        column = [motif[i] for motif in motifs]
        most_common_count = Counter(column).most_common(1)[0][1]
        mismatch_score += len(motifs) - most_common_count
        
    normalized_score = mismatch_score / k
    return normalized_score

def get_consensus_sequence(profile):
    #Derives consensus sequence from a profile.
    consensus = ""
    k = len(profile['A'])
    for i in range(k):
        max_prob = -1.0
        consensus_char = ''
        for nucleotide in 'ACGT':
            if profile[nucleotide][i] > max_prob:
                max_prob = profile[nucleotide][i]
                consensus_char = nucleotide
        consensus += consensus_char
    return consensus

def calculate_information_content(profile, background_freq={'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25}):
    #Calculates the Information Content (IC) of a PWM.
    if not profile or not profile.get('A'): # Check if profile is empty
        return 0.0

    k = len(profile['A'])
    total_ic = 0.0

    for i in range(k): # For each position in the motif
        position_ic = 0.0
        for base in 'ACGT':
            prob_at_pos = profile[base][i]
            bg_prob = background_freq.get(base, 0.25) # Default to 0.25 if base not in background

            if prob_at_pos > 0 and bg_prob > 0: # Avoid log(0) and division by zero
                position_ic += prob_at_pos * math.log2(prob_at_pos / bg_prob)
        total_ic += position_ic
        
    return total_ic

#Gibbs Sampler Algorithm

def gibbs_sampler(sequences, k, n_iter=1000):
    num_sequences = len(sequences)

    initial_motifs = []
    for seq in sequences:
        if len(seq) < k:
            print(f"Warning: Sequence '{seq[:20]}...' is shorter than motif length k={k}.")
            return None, float('inf'), 0.0 # Also return 0 IC
        start_index = random.randint(0, len(seq) - k)
        initial_motifs.append(seq[start_index : start_index + k])
    
    motifs = list(initial_motifs)
    best_motifs = list(motifs)
    best_score = score_motifs(motifs)
    
    # Track the best PWM for IC calculation later
    best_pwm_for_ic = create_profile(best_motifs)

    for _ in range(n_iter):
        i = random.randint(0, num_sequences - 1)
        
        motifs_except_i = motifs[:i] + motifs[i+1:]
        profile = create_profile(motifs_except_i)
        
        new_motif = sample_kmer(sequences[i], k, profile)
        
        if new_motif is not None:
            motifs[i] = new_motif

        current_score = score_motifs(motifs)
        if current_score < best_score:
            best_score = current_score
            best_motifs = list(motifs)
            best_pwm_for_ic = create_profile(best_motifs) # Update PWM for best motifs

    # Calculate IC for the final best motif set
    final_best_pwm = create_profile(best_motifs)
    information_content = calculate_information_content(final_best_pwm)

    return best_motifs, best_score, information_content

# Execution

def run_motif_discovery(file_path, k_range=range(3, 8), n_iter=1000):

    sequences = read_fasta(file_path)

    if not sequences:
        print("No sequences loaded. Exiting motif discovery.")
        return None

    overall_best_mismatch_score = float('inf') # For comparison based on the original score
    overall_best_pwm_by_score = None
    overall_best_k_by_score = None
    overall_best_consensus_by_score = None

    overall_best_ic = -1.0 # For comparison based on information Content
    overall_best_pwm_by_ic = None
    overall_best_k_by_ic = None
    overall_best_consensus_by_ic = None


    for k in k_range:
        print(f"\n--- Running Gibbs Sampler for k = {k} ---")
        num_runs_per_k = 5
        
        best_motifs_for_current_k = None
        best_score_for_current_k = float('inf')
        best_ic_for_current_k = -1.0

        for run_idx in range(num_runs_per_k):
            motifs, score, ic = gibbs_sampler(sequences, k, n_iter=n_iter)
            
            if motifs is not None:
                # Update best for current k based on score
                if score < best_score_for_current_k:
                    best_score_for_current_k = score
                    best_motifs_for_current_k = motifs
                    best_ic_for_current_k = ic 
                
                # Update best for current k based on IC
                if ic > best_ic_for_current_k: 
                    best_ic_for_current_k = ic
        
        if best_motifs_for_current_k:
            print(f"  Best normalized mismatch score for k={k} across {num_runs_per_k} runs: {best_score_for_current_k:.3f}")
            print(f"  Highest Information Content for k={k} across {num_runs_per_k} runs: {best_ic_for_current_k:.3f} bits")
            
            current_pwm_from_best_score = create_profile(best_motifs_for_current_k)
            current_consensus_from_best_score = get_consensus_sequence(current_pwm_from_best_score)

            if best_score_for_current_k < overall_best_mismatch_score:
                overall_best_mismatch_score = best_score_for_current_k
                overall_best_pwm_by_score = current_pwm_from_best_score
                overall_best_k_by_score = k
                overall_best_consensus_by_score = current_consensus_from_best_score
            
            if best_ic_for_current_k > overall_best_ic: 
                overall_best_ic = best_ic_for_current_k
                overall_best_pwm_by_ic = current_pwm_from_best_score 
                overall_best_k_by_ic = k
                overall_best_consensus_by_ic = current_consensus_from_best_score
        else:
            print(f"  Skipping k={k} due to sequence length issues or no valid motifs found.")

    # Results
    print("\n" + "="*50)
    print("--- Results Based on Normalized Mismatch Score (Lower is Better) ---")
    if overall_best_pwm_by_score:
        print(f"Overall Best Motif (k={overall_best_k_by_score}) found with score: {overall_best_mismatch_score:.3f}")
        print(f"Consensus Sequence: {overall_best_consensus_by_score}")
        print("\nPosition Weight Matrix (PWM - Frequencies):")
        print("Pos\tA\tC\tG\tT")
        for i in range(overall_best_k_by_score):
            print(f"{i+1}\t" + "\t".join(f"{overall_best_pwm_by_score[base][i]:.3f}" for base in "ACGT"))
    else:
        print("No suitable motifs found based on normalized mismatch score.")

    print("\n" + "="*50)
    print("--- Results Based on Information Content (Higher is Better) ---")
    if overall_best_pwm_by_ic:
        print(f"Overall Best Motif (k={overall_best_k_by_ic}) found with Information Content: {overall_best_ic:.3f} bits")
        print(f"Consensus Sequence: {overall_best_consensus_by_ic}")
        print("\nPosition Weight Matrix (PWM - Frequencies):")
        print("Pos\tA\tC\tG\tT")
        for i in range(overall_best_k_by_ic):
            print(f"{i+1}\t" + "\t".join(f"{overall_best_pwm_by_ic[base][i]:.3f}" for base in "ACGT"))
    else:
        print("No suitable motifs found based on Information Content.")
    print("="*50)

if __name__ == "__main__":
    fasta_file_path = "motifs_in_sequence.fa"
    run_motif_discovery(fasta_file_path)


--- Running Gibbs Sampler for k = 3 ---
  Best normalized mismatch score for k=3 across 5 runs: 8.333
  Highest Information Content for k=3 across 5 runs: 2.982 bits

--- Running Gibbs Sampler for k = 4 ---
  Best normalized mismatch score for k=4 across 5 runs: 14.750
  Highest Information Content for k=4 across 5 runs: 2.768 bits

--- Running Gibbs Sampler for k = 5 ---
  Best normalized mismatch score for k=5 across 5 runs: 11.600
  Highest Information Content for k=5 across 5 runs: 4.624 bits

--- Running Gibbs Sampler for k = 6 ---
  Best normalized mismatch score for k=6 across 5 runs: 11.000
  Highest Information Content for k=6 across 5 runs: 6.090 bits

--- Running Gibbs Sampler for k = 7 ---
  Best normalized mismatch score for k=7 across 5 runs: 14.000
  Highest Information Content for k=7 across 5 runs: 5.661 bits

--- Results Based on Normalized Mismatch Score (Lower is Better) ---
Overall Best Motif (k=3) found with score: 8.333
Consensus Sequence: GAT

Position Weight M